In [ ]:
################### TO BE RUN ON COLAB DUE TO COMPUTE CONSTRAINTS
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, average_precision_score, classification_report
from sklearn.metrics import roc_curve, precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

import random


In [ ]:

from google.colab import files
uploaded = files.upload()


In [ ]:

# Constants
random_state = 14
n_combinations = 30
early_stopping_rounds = 50

# Load data
df = pd.read_parquet('model_inputs.parquet')

# Identify categorical columns
categorical_cols = [
    col for col in df.columns
    if ('Code_' in col or '_cluster' in col) and not (col.endswith('_amount') or col.endswith('_rate'))
]


In [ ]:

X = df.drop(columns=['SUBJECT_ID', 'UCR_Flag'])
y = df['UCR_Flag']

X.columns = X.columns.str.replace(r'[^\w]', '_', regex=True)

X = X.loc[:, ~X.columns.duplicated(keep='first')]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

param_grid = {
    'n_estimators': [500],  
    'learning_rate': [0.01, 0.05],  
    'num_leaves': [31, 63, 127, 254],  
    'max_depth': [-1, 5, 10, 20], 
    'min_child_samples': [10, 20, 30],  
    'subsample': [0.7, 0.9], 
    'colsample_bytree': [0.7, 0.9],  
    'reg_alpha': [0.1, 0.5, 1.0],  
    'reg_lambda': [0.1, 0.5, 1.0],  
    'scale_pos_weight': [300, 600, 900]  
}


In [ ]:

def sample_param(param_grid):
    return {key: random.choice(values) for key, values in param_grid.items()}
         
def train_model_with_params(params):
    fold_precisions = []
    fold_recalls = []
    fold_pr_aucs = []

    for train_index, val_index in kf.split(X_train, y_train):

        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        train_fold = pd.concat([X_train_fold, y_train_fold], axis=1)
        
        majority_class = train_fold[train_fold['UCR_Flag'] == 0]
        minority_class = train_fold[train_fold['UCR_Flag'] == 1]

        #Upsample the minority class
        minority_upsampled = resample(
            minority_class,
            replace=True,
            n_samples=(len(majority_class) // 10), 
            random_state=random_state
        )

        #Combine upsampled minority with majority
        upsampled_train_fold = pd.concat([majority_class, minority_upsampled])

        #Separate features and target for training
        X_train_fold_upsampled = upsampled_train_fold.drop(columns=['UCR_Flag'])
        y_train_fold_upsampled = upsampled_train_fold['UCR_Flag']

        #Initialize the model
        model = lgb.LGBMClassifier(
            **params,
            random_state=random_state,
            max_bin=127,
            n_jobs=-1
        )

        # Callbacks for early stopping
        callbacks = [
            lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=10)  # Log evaluation every 10 rounds
        ]

        #Train
        model.fit(
            X_train_fold_upsampled,
            y_train_fold_upsampled,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric='average_precision',
            categorical_feature=categorical_cols,
            callbacks=callbacks
        )

        #Evaluate
        y_val_pred = model.predict(X_val_fold)
        y_val_prob = model.predict_proba(X_val_fold)[:, 1]

        precision = precision_score(y_val_fold, y_val_pred, zero_division=0)
        recall = recall_score(y_val_fold, y_val_pred, zero_division=0)
        pr_auc = average_precision_score(y_val_fold, y_val_prob)

        fold_precisions.append(precision)
        fold_recalls.append(recall)
        fold_pr_aucs.append(pr_auc)

    return params, np.mean(fold_precisions), np.mean(fold_recalls), np.mean(fold_pr_aucs)




In [ ]:
overall_best_pr_auc = 0
overall_best_params = None
fold_results = []

progress = tqdm(range(n_combinations), desc='Randomized Search', unit='iteration', dynamic_ncols=True)

with ThreadPoolExecutor(max_workers=1) as executor:  # Adjust max_workers to control memory usage
    sampled_params = [sample_param(param_grid) for _ in range(n_combinations)]
    futures = [executor.submit(train_model_with_params, params) for params in sampled_params]

    for future in as_completed(futures):
        try:
            params, precision, recall, pr_auc = future.result()
        except Exception as e:
            print(f"Error during training: {e}")
            continue

        if pr_auc > overall_best_pr_auc:
            overall_best_pr_auc = pr_auc
            overall_best_params = params
            
        progress.set_postfix({
            'Best PR-AUC (Avg Precision)': overall_best_pr_auc,
            'Best Params': overall_best_params
        })
        progress.update(1)

    progress.close()

# Output the best parameters and accuracy
print(f"Best PR-AUC (Avg Precision): {overall_best_pr_auc}")
print(f"Best Parameters: {overall_best_params}")



In [ ]:
desired_params = {
    'n_estimators': 1500,  
    'learning_rate': 0.05,  
    'num_leaves': 127,  
    'max_depth': -1, 
    'min_child_samples': 10,  
    'subsample': 0.7, 
    'colsample_bytree': 0.9,  
    'reg_alpha': 0.5,  
    'reg_lambda': 0.1,  
    'scale_pos_weight': 900
}



In [ ]:
train_data = pd.concat([X_train, y_train], axis=1)

majority_class = train_data[train_data['UCR_Flag'] == 0]
minority_class = train_data[train_data['UCR_Flag'] == 1]

minority_upsampled = resample(
    minority_class,
    replace=True,
    n_samples=(len(majority_class) // 10),  # Adjust the ratio as necessary
    random_state=random_state
)

upsampled_data = pd.concat([majority_class, minority_upsampled])

X_train_upsampled = upsampled_data.drop(columns=['UCR_Flag'])
y_train_upsampled = upsampled_data['UCR_Flag']

initial_model = lgb.LGBMClassifier(
    **desired_params,
    random_state=random_state,
    max_bin=127,
    n_jobs=-1
)

callbacks = [
    lgb.early_stopping(stopping_rounds=200, verbose=True)
]

initial_model.fit(
    X_train_upsampled,
    y_train_upsampled,
    eval_set=[(X_test, y_test)],
    eval_metric='average_precision',
    callbacks=callbacks,
    categorical_feature=categorical_cols
)

y_test_pred = initial_model.predict(X_test)
y_test_proba = initial_model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_test_pred, zero_division=0)
recall = recall_score(y_test, y_test_pred, zero_division=0)
pr_auc = average_precision_score(y_test, y_test_proba)
accuracy = accuracy_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_proba)

# Display evaluation metrics
print(f"Test Accuracy: {accuracy}")
print(f"Test AUC: {auc}")
print(f"Test PR-AUC: {pr_auc}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(classification_report(y_test, y_test_pred))



In [ ]:
predictions_df = pd.DataFrame({
    'Predicted_Probability': y_test_proba,
    'True_Label': y_test
})


metrics = {
    "Metric": ["Accuracy", "AUC", "PR-AUC", "Precision", "Recall"],
    "Value": [accuracy, auc, pr_auc, precision, recall]
}

metrics_df = pd.DataFrame(metrics)
print("Model Evaluation Metrics:")
print(metrics_df)



In [ ]:
# Plot ROC Curve
plt.figure(figsize=(8, 6))
roc_display = RocCurveDisplay.from_estimator(initial_model, X_test, y_test)
plt.title("ROC Curve")
plt.show()




In [ ]:
# Plot Precision-Recall Curve
plt.figure(figsize=(8, 6))
pr_display = PrecisionRecallDisplay.from_estimator(initial_model, X_test, y_test)
plt.title("Precision-Recall Curve")
plt.show()
